In [1]:
!pip install kgbench-loader

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 50.5 MB/s eta 0:00:00
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=64cd916e38af8798c474f6c405b5927bb1768bd746afc71aa98a85941fb0f3c6
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.0 which is inco

In [1]:
!pip install torch_geometric
import torch
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from torch_geometric.utils import degree
import kgbench as kg

# 2. Load Data (and apply your Dummy Fix if needed)
data = kg.load('dmg777k', torch=True, final=True)
if not hasattr(data, 'i2n') or data.i2n is None:
    data.i2n = [("dummy", "none")] * data.num_entities



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.0 MB/s eta 0:00:00
loaded data dmg777k (53.5s).


In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from torch_geometric.utils import to_undirected

# 1. Setup & Safety Checks
num_nodes = data.num_entities
# Ensure edge_index is within bounds
mask_valid_edges = (edge_index[0] < num_nodes) & (edge_index[1] < num_nodes)
edge_index = edge_index[:, mask_valid_edges]
edge_type = edge_type[mask_valid_edges]

print(f"Graph: {num_nodes} nodes, {edge_index.size(1)} edges.")

# 2. FEATURE ENGINEERING: Bag-of-Relations (BoR)
# Instead of random noise, we use the edge types connected to each node as features.
# This is physically grounded and much better than random noise.
def get_bor_features(num_nodes, edge_index, edge_type, num_relations):
    # Create a feature matrix where x[i, r] = 1 if node i has edge type r
    # We use scatter_add to count occurrences
    x = torch.zeros(num_nodes, num_relations, device=edge_index.device)

    # Incoming edges
    # target nodes are at edge_index[1]
    target_nodes = edge_index[1]

    # Create one-hot for relations
    rel_one_hot = F.one_hot(edge_type, num_classes=num_relations).float()

    # Aggregate: Scatter sum relation vectors into target nodes
    x.scatter_add_(0, target_nodes.unsqueeze(1).expand(-1, num_relations), rel_one_hot)

    # Normalize (Log count or boolean)
    x = torch.log1p(x)
    return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
edge_index = edge_index.to(device)
edge_type = edge_type.to(device)

# Generate BETTER features
x_features = get_bor_features(num_nodes, edge_index, edge_type, num_relations).to(device)
print(f"Generated Bag-of-Relations Features: {x_features.size()}")

# 3. Model: Standard RGCN (Not Randomized anymore, since we have BoR features)
class BoR_RGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations):
        super().__init__()
        # Layer 1: BoR Features -> Hidden
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations, num_bases=30)
        # Layer 2: Hidden -> Class
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations, num_bases=30)
        self.dropout = 0.5

    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, edge_type)
        return F.log_softmax(x, dim=1)

# 4. Train
model = BoR_RGCN(num_relations, 64, data.num_classes, num_relations).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Prepare Labels
node_labels = torch.full((num_nodes,), -1, dtype=torch.long).to(device)
node_labels[data.training[:, 0]] = data.training[:, 1].long().to(device)
node_labels[data.withheld[:, 0]] = data.withheld[:, 1].long().to(device)
train_mask = data.training[:, 0].to(device)
test_mask = data.withheld[:, 0].to(device)

print("🚀 Starting Training with Bag-of-Relations...")
for epoch in range(101):
    model.train()
    optimizer.zero_grad()
    out = model(x_features, edge_index, edge_type)
    loss = F.nll_loss(out[train_mask], node_labels[train_mask])
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        pred = out.argmax(dim=1)
        test_acc = (pred[test_mask] == node_labels[test_mask]).sum().item() / test_mask.size(0)
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Test Acc: {test_acc:.4f}")


Graph: 341270 nodes, 777124 edges.
Generated Bag-of-Relations Features: torch.Size([341270, 60])
🚀 Starting Training with Bag-of-Relations...
Epoch 000 | Loss: 1.6507 | Test Acc: 0.1754
Epoch 010 | Loss: 1.2477 | Test Acc: 0.5007
Epoch 020 | Loss: 1.2072 | Test Acc: 0.5067
Epoch 030 | Loss: 1.1994 | Test Acc: 0.5007
Epoch 040 | Loss: 1.1951 | Test Acc: 0.5022
Epoch 050 | Loss: 1.1954 | Test Acc: 0.4953
Epoch 060 | Loss: 1.1896 | Test Acc: 0.4948
Epoch 070 | Loss: 1.1914 | Test Acc: 0.4973
Epoch 080 | Loss: 1.1876 | Test Acc: 0.5022
Epoch 090 | Loss: 1.1897 | Test Acc: 0.4958
Epoch 100 | Loss: 1.1881 | Test Acc: 0.4983


In [19]:
# Re-create loaders with num_workers=0 to bypass multiprocessing import errors
print("⏳ Re-initializing NeighborLoader with num_workers=0...")

train_loader = NeighborLoader(
    clean_data, # Ensure you use the 'clean_data' object we created earlier
    num_neighbors=[25, 10],
    batch_size=2048,
    input_nodes=data.training[:, 0].cpu(),
    shuffle=True,
    num_workers=0  # <--- CRITICAL FIX: Run in main process
)

test_loader = NeighborLoader(
    clean_data,
    num_neighbors=[25, 10],
    batch_size=4096,
    input_nodes=data.withheld[:, 0].cpu(),
    shuffle=False,
    num_workers=0  # <--- CRITICAL FIX
)

print(f"✅ Loaders ready (Main Process). Train batches: {len(train_loader)}")


⏳ Re-initializing NeighborLoader with num_workers=0...
✅ Loaders ready (Main Process). Train batches: 3


In [21]:
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.5.1+cu121.html


Looking in links: https://data.pyg.org/whl/torch-2.5.1+cu121.html


In [2]:
# Gather some statistics about the graph.
print(f'Number of nodes: {data.num_entities}')
print(f"Relations (Edge Types): {data.num_relations}")
print(f"Total Triples (Edges): {len(data.triples)}")
print(f"Classes: {data.num_classes}")

Number of nodes: 341270
Relations (Edge Types): 60
Total Triples (Edges): 777124
Classes: 5


In [3]:
print(data.withheld)
print(len(data.i2r))
for i in range(5):
  print(data.i2r[i])
print("\n")
print(len(data.triples))
for i in range(10):
  print(data.triples[i])

tensor([[286544,      1],
        [286290,      1],
        [284598,      1],
        ...,
        [284792,      4],
        [285234,      1],
        [284104,      1]], dtype=torch.int32)
60
http://data.pdok.nl/def/pdok#asWKT-RD
http://dbpedia.org/ontology/city
http://dbpedia.org/ontology/codeNationalMonument
http://dbpedia.org/ontology/location
http://dbpedia.org/ontology/name


777124
tensor([130685,     28,  54795])
tensor([130685,     31, 201822])
tensor([130690,     28,  58948])
tensor([130690,     31, 201822])
tensor([130691,     28,  63024])
tensor([130691,     31, 201822])
tensor([130693,     28,  53243])
tensor([130693,     31, 201822])
tensor([130699,     28,  60610])
tensor([130699,     31, 201822])


In [27]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from torch_geometric.utils import k_hop_subgraph
import gc

# ==========================================
# 1. SETUP & CLEANUP
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Cleanup memory
try:
    del model, optimizer
except:
    pass
torch.cuda.empty_cache()
gc.collect()

# Ensure we have CPU tensors for sampling
# (Assuming 'x_features' and 'clean_data' exist from previous steps)
# If x_features doesn't exist, we re-create the Bag-of-Relations features quickly:
if 'x_features_cpu' not in locals():
    print("Regenerating CPU features...")
    # Just in case x_features was overwritten
    def get_bor_features_cpu(num_nodes, edge_index, edge_type, num_relations):
        x = torch.zeros(num_nodes, num_relations)
        target_nodes = edge_index[1]
        rel_one_hot = F.one_hot(edge_type, num_classes=num_relations).float()
        x.scatter_add_(0, target_nodes.unsqueeze(1).expand(-1, num_relations), rel_one_hot)
        return torch.log1p(x)

    x_features_cpu = get_bor_features_cpu(
        clean_data.num_nodes,
        clean_data.edge_index,
        clean_data.edge_attr,
        num_relations
    )

# ==========================================
# 2. DEFINE DEEP SOTA MODEL
# ==========================================
class SOTA_DeepRGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, num_layers=3):
        super().__init__()

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()

        # Using num_bases=30 for regularization and parameter efficiency
        bases = 30

        # Layer 1: Input -> Hidden
        self.convs.append(RGCNConv(in_channels, hidden_channels, num_relations, num_bases=bases))
        self.bns.append(torch.nn.BatchNorm1d(hidden_channels))

        # Middle Layers (Residual Blocks)
        for _ in range(num_layers - 2):
            self.convs.append(RGCNConv(hidden_channels, hidden_channels, num_relations, num_bases=bases))
            self.bns.append(torch.nn.BatchNorm1d(hidden_channels))

        # Output Layer
        self.convs.append(RGCNConv(hidden_channels, out_channels, num_relations, num_bases=bases))

        self.dropout = 0.5

    def forward(self, x, edge_index, edge_type):
        # Layer 1
        x = self.convs[0](x, edge_index, edge_type)
        x = self.bns[0](x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Middle Layers with Residual Connections
        for i in range(1, len(self.convs) - 1):
            x_in = x
            x = self.convs[i](x, edge_index, edge_type)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            # Residual connection
            x = x + x_in

        # Output Layer
        x = self.convs[-1](x, edge_index, edge_type)
        return F.log_softmax(x, dim=1)

# Initialize Model
hidden_dim = 128
model = SOTA_DeepRGCN(num_relations, hidden_dim, data.num_classes, num_relations, num_layers=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# ==========================================
# 3. MANUAL MINI-BATCH TRAINING LOOP
# ==========================================
print(f"🚀 Starting SOTA Manual Training (Deep R-GCN, 128 dim)...")

for epoch in range(51): # 50 Epochs
    model.train()

    # Shuffle training nodes manually
    perm = torch.randperm(data.training.size(0))
    train_nodes = data.training[:, 0].cpu()
    shuffled_nodes = train_nodes[perm]

    total_loss = 0
    total_examples = 0

    # Iterate in chunks
    batch_size = 2048
    for i in range(0, len(shuffled_nodes), batch_size):
        optimizer.zero_grad()

        seeds = shuffled_nodes[i:i+batch_size]

        # Sample k-hop subgraph (CPU)
        subset, sub_edge_index, mapping, edge_mask = k_hop_subgraph(
            seeds, num_hops=2, edge_index=clean_data.edge_index, relabel_nodes=True
        )

        # --- FIX IS HERE ---
        # 1. Slice features on CPU first (because x_features_cpu and subset are both on CPU)
        batch_x = x_features_cpu[subset].to(device)

        # 2. Now move structure to GPU
        sub_edge_index = sub_edge_index.to(device)
        sub_edge_type = clean_data.edge_attr[edge_mask].to(device)
        # -------------------

        # Forward
        out = model(batch_x, sub_edge_index, sub_edge_type)

        # Loss
        out_seeds = out[mapping]
        target = node_labels_cpu[seeds].to(device)

        loss = F.nll_loss(out_seeds, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(seeds)
        total_examples += len(seeds)

    avg_loss = total_loss / total_examples

    # Evaluation
    if epoch % 5 == 0:
        model.eval()
        total_correct = 0
        total_test = 0

        test_nodes = data.withheld[:, 0].cpu()

        with torch.no_grad():
            for i in range(0, len(test_nodes), 4096):
                seeds = test_nodes[i:i+4096]

                subset, sub_edge_index, mapping, edge_mask = k_hop_subgraph(
                    seeds, num_hops=2, edge_index=clean_data.edge_index, relabel_nodes=True
                )

                # Apply same fix for test loop
                batch_x = x_features_cpu[subset].to(device)
                sub_edge_index = sub_edge_index.to(device)
                sub_edge_type = clean_data.edge_attr[edge_mask].to(device)

                out = model(batch_x, sub_edge_index, sub_edge_type)
                pred = out[mapping].argmax(dim=1)
                target = node_labels_cpu[seeds].to(device)

                total_correct += (pred == target).sum().item()
                total_test += len(seeds)

        acc = total_correct / total_test
        print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}")

print("✅ Training Complete.")


Using device: cuda
🚀 Starting SOTA Manual Training (Deep R-GCN, 128 dim)...
Epoch 000 | Loss: 3.6732 | Test Acc: 0.1349
Epoch 005 | Loss: 1.7853 | Test Acc: 0.4458
Epoch 010 | Loss: 1.3520 | Test Acc: 0.4853
Epoch 015 | Loss: 1.2586 | Test Acc: 0.4898
Epoch 020 | Loss: 1.2297 | Test Acc: 0.4998
Epoch 025 | Loss: 1.2263 | Test Acc: 0.5007
Epoch 030 | Loss: 1.2227 | Test Acc: 0.5047
Epoch 035 | Loss: 1.2153 | Test Acc: 0.5037
Epoch 040 | Loss: 1.2210 | Test Acc: 0.4968
Epoch 045 | Loss: 1.2017 | Test Acc: 0.4968
Epoch 050 | Loss: 1.2076 | Test Acc: 0.4923
✅ Training Complete.
